1. Imports, configurações iniciais e funções

In [ ]:
# === CÉLULA 1: imports, configurações e utilitários ===

import pandas as pd
import requests_cache
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Configurações Nominatim (boas práticas)
USER_AGENT = "insercao-urbana-mcmv/1.0 (seu.email@provedor.com.br)"
RATE_LIMIT_SECONDS = 1.0 # respeitar limite (>= 1 req/s)

#Colunas obrigatórias usadas na etapa 2
COLUNAS_OBRIGATORIAS = ["cod_operacao", "txt_nome_empreendimento", "txt_endereco", "txt_nome_municipio", "txt_sigla_uf", "txt_modalidade", "lat", "long"]

def init_geocoder():
  """Inicializa cache global + Nominatim + RateLimiter."""
  requests_cache.install_cache(
      cache_name="geocoding_cache",
      backend="sqlite",
      expire_after=60*60*24*7 # 7 dias
  )
  geolocator = Nominatim(user_agent=USER_AGENT, timeout=15)
  geocode = RateLimiter(geolocator.geocode, min_delay_seconds=RATE_LIMIT_SECONDS, swallow_exceptions=True)
  return geocode

def validar_latlon_str(txt):
  """Valida string 'lat,lon' e retorna (lat, lon) se ok; caso contrário, None."""
  if not txt or "," not in txt:
    return None
  
  try:
    a, b = txt.split(",", 1)
    lat = float(a.strip().replace(",", "."))
    lon = float(b.strip().replace(",", "."))
    if -90 <= lat <= 90 and -180 <= lon <= 180:
      return lat, lon
  except Exception:
    pass
  return None

def geocode_endereco(geocode, endereco, cidade, uf):
  """
  Geocodifica combinando 'endereco, cidade, uf, Brasil' para reduzir ambiguidade.
  Retorna (lat,lon) se encontrar ou (None, None) se não encontrar.
  """

  # Proteção contra endereço "vazio"
  if not endereco or endereco.strip() =="":
    return None, None
  
  query = f"{endereco}, {cidade}, {uf}, Brasil"
  loc = geocode(query)
  if loc is None:
    return None, None
  return loc.latitude, loc.longitude

2. Leitura do CSV

In [ ]:
# === LEITURA SIMPLES DO CSV (SEPARADOR ';' E ENCODING 'utf-8') ===

csv_path = input("Digite o caminho do arquivo CSV: ").strip()


df = pd.read_csv(
    csv_path,
    dtype=str,
    sep=";",          # <<<<< SEPARADOR DO ARQUIVO .CSV
    encoding="utf-8", # <<<<< ENCODING DO ARQUIVO .CSV
    engine="python"   # <<<<< A MAIS TOLERANTE A PEQUENAS INCONSISTÊNCIAS
)

df.columns = [c.strip() for c in df.columns]

faltantes = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if faltantes:
  print("\n CSV inválido! Faltam colunas obrigatórias:", faltantes)
  print("Colunas encontradas:", list(df.columns))
  raise SystemExit

df = df[COLUNAS_OBRIGATORIAS].copy()

# Limpeza geral
for c in ["cod_operacao", "txt_nome_empreendimento", "txt_endereco", "txt_nome_municipio", "txt_sigla_uf", "txt_modalidade"]:
  df[c] = df[c].astype(str).fillna("").str.strip()

df["lat"] = df["lat"].astype(str).fillna("").str.strip()
df["long"] = df["long"].astype(str).fillna("").str.strip()

print("\nPrévia das primeiras linhas:\n")
print(df.head())

3. Listar municípios e selecionar um

In [ ]:
# CÉLULA 3: Listar cidades/UF disponíveis e escolher por índice

munics = (df[["txt_nome_municipio", "txt_sigla_uf"]].drop_duplicates().sort_values(["txt_nome_municipio", "txt_sigla_uf"]).reset_index(drop=True))

print("\nMunicípios encontrados no CSV:\n")
for idx, row in munics.iterrows():
  print(f"[{idx}] {row['txt_nome_municipio']} / {row['txt_sigla_uf']}")

# Loop de seleção até o usuário informar um índice válido
while True:
  entrada = input("\nDigite o ÍNDICE da cidade/UF que deseja processar: ").strip()

  try:
    idx_sel = int(entrada)
  except ValueError:
    print("Entrada inválida. Digite um número inteiro.")
    continue

  if idx_sel < 0 or idx_sel >= len(munics):
    print(f"Índice fora do intervalo válido (0 a {len(munics) - 1}). Tente novamente.")
    continue

  # Se chegou aqui, o índice é válido
  break

cidade_sel = munics.loc[idx_sel, "txt_nome_municipio"]
uf_sel     = munics.loc[idx_sel, "txt_sigla_uf"]

# Resultado final

df_munic = df[(df["txt_nome_municipio"] == cidade_sel) & (df["txt_sigla_uf"] == uf_sel)].copy()

print(f"\nRegistros encontrados para {cidade_sel}/{uf_sel}: {len(df_munic)}")
print(df_munic.head())

4. Geocodificação

In [ ]:
# CÉLULA 4: geocodificação + correção de endereço ou lat/long ===

geocode = init_geocoder()

def precisa_geocod(row):
  lat = (row.get("lat") or "").strip()
  lon = (row.get("long") or "").strip()
  if lat == "" or lat.lower() in ["nan", "none"]:
    return True
  if lon == "" or lon.lower() in ["nan", "none"]:
    return True
  return False

print("\nIniciando geocodificação dos registros que NÃO possuem lat/long...\n")

for idx in df_munic.index:
  # se já tem coordenadas, pula
  if not precisa_geocod(df_munic.loc[idx]):
    continue

  while True:
    # Recarrega SEMPRE a linha atual a partir do DF (garante que prints e geocod usem o valor mais recente)
    row = df_munic.loc[idx]

    endereco_atual = (row["txt_endereco"] or "").strip()

    print(f"\n--- OPERAÇÃO {row['cod_operacao']} | {row['txt_nome_empreendimento']} ---")
    print(f"Endereço atual: {endereco_atual} - {row['txt_nome_municipio']}/{row['txt_sigla_uf']}")

    #tentativa automática com o endereço ATUAL do DF
    lat, lon = geocode_endereco(
        geocode,
        row["txt_endereco"],
        row["txt_nome_municipio"],
        row["txt_sigla_uf"]
    )

    if lat is not None and lon is not None:
      print(f"Geocodificado automaticamente: {lat}, {lon}")
      df_munic.at[idx, "lat"] = f"{lat: .8f}"
      df_munic.at[idx, "long"] = f"{lon: .8f}"
      break # vai para o próximo registro

    # 2) não encontrou >>> oferecer opções
    print("Não foi possível geocodificar automaticamente.")
    print("Escolha uma opção:")
    print(" (1) Informar um NOVO ENDEREÇO (apenas 'rua + número', sem cidade/UF) para tentar novamente")
    print(" (2) Informar manualmente 'lat,lon'")
    print(" (ENTER) Pular este registro (deixar sem coordenadas)")

    escolha = input("Sua escolha [1/2/ENTER]: ").strip()

    if escolha == "1":
      novo_end = input("Informe um NOVO endereço (apenas 'rua + número', sem cidade/UF): ").strip()
      if novo_end == "":
        print("Endereço vazio. Nenhuma alteração realizada.")
        #volta ao topo do while para oferecer opções novamente
        continue

      # Atualiza APENAS o campo 'endereco' no DF
      df_munic.at[idx, "txt_endereco"] = novo_end
      print(f"Endereço atualizado para: '{novo_end}' - {row['txt_nome_municipio']}/{row['txt_sigla_uf']}")
      #volta ao topo do while; row será recarregada e a tentativa automática sreá repetida
      continue

    elif escolha == "2":
      entrada = input("Digite lat,lon (ex.: -23.55052,-46.63331): ").strip()
      if entrada =="":
        print("Nenhuma coordenada informada. Retornando ao menu de opções...")
        continue

      coords = validar_latlon_str(entrada)
      if coords:
        lat, lon = coords
        df_munic.at[idx, "lat"] = f"{lat: .8f}"
        df_munic.at[idx, "long"] = f"{lon: .8f}"
        print(f"Coordenadas salvas: {lat}, {lon}")
        break # segue para o próximo registro

      else:
        print("Entrada inválida. Tente novamente ou escolha outra opção.")

    else:
      #Enter >>> pular
      print("Registro deixado para corrigir depois.")
      break # sai do while sem preencher coordenadas; ficará para exclusão no final



5. CSV de saída apenas com lat/long válidos e outro CSV em separado com os "excluídos"

In [ ]:
# === CÉLULA 5: salvar apenas linhas com coordenadas preenchidas, com timestamp ===

import os
from datetime import datetime

def tem_coord_ok(row):
  lat = (row.get("lat") or "").strip()
  lon = (row.get("long") or "").strip()
    #Considera válido se ambos não estiverem vazios/nulos
  return (lat not in ["", "nan", "none", "NaN", "None"]) and (lon not in ["", "nan", "none", "NaN", "None"])

# Centralizar saídas em uma pasta:
OUT_DIR = "C:/pmcmv/saidas_geocoding"
os.makedirs(OUT_DIR, exist_ok=True)

#Filtra válidos e excluídos
df_validos = df_munic[df_munic.apply(tem_coord_ok, axis=1)].copy()
df_excluidos = df_munic[~df_munic.apply(tem_coord_ok, axis=1)].copy()

#Gera timestamp
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Monta nomes dos arquivos (com cidade/UF e timestamp)
base_validos = f"avaliacao_validos_{cidade_sel}_{uf_sel}_{ts}.csv".replace(" ", "_")
base_excluidos = f"avaliacao_excluidos_(cidade_sel)_{uf_sel}_{ts}.csv".replace(" ", "_")

path_validos = os.path.join(OUT_DIR, base_validos)
path_excluidos = os.path.join(OUT_DIR, base_excluidos)

#Salva "válidos"
df_validos.to_csv(path_validos, index=False, encoding="utf-8", sep=";")

#Salva "excluídos" somente se existir pelo menos um
if not df_excluidos.empty:
  df_excluidos.to_csv(path_excluidos, index=False, encoding="utf-8", sep=";")
  print(f"Excluídos salvos: {path_excluidos} (linhas: {len(df_excluidos)}")
else:
  print("Não há excluídos; todos os registros têm coordenadas.")

print(f"Válidos salvos: {path_validos} (linhas: {len(df_validos)})")